In [6]:
import pandas as pd
from sklearn.metrics import cohen_kappa_score, precision_score, recall_score, confusion_matrix

df = pd.read_csv("./human_oracle_cross_validation.csv")

# -----------------------------
# 1. Copy dataframe
# -----------------------------
df_eval = df.copy()

# -----------------------------
# 2. Clean + convert labels safely
# -----------------------------
df_eval["source_query_project"] = pd.to_numeric(
    df_eval["source_query_project"],
    errors="coerce"
)

df_eval["cross_val_category"] = pd.to_numeric(
    df_eval["cross_val_category"],
    errors="coerce"
)

# -----------------------------
# 3. Keep only valid classes
# -----------------------------
valid_labels = {1, 2, 3, 4}

df_eval = df_eval[
    df_eval["source_query_project"].isin(valid_labels) &
    df_eval["cross_val_category"].isin(valid_labels)
].copy()

# -----------------------------
# 4. Sanity check (IMPORTANT)
# -----------------------------
print(f"Rows used for evaluation: {len(df_eval)}")

if len(df_eval) == 0:
    raise ValueError(
        "No valid rows left after filtering. "
        "Check label formats in your dataframe."
    )

# -----------------------------
# 5. Map to binary classes
# -----------------------------
# 1,2 → Positive (1)
# 3,4 → Negative (0)

label_map = {
    1: 1,
    2: 1,
    3: 0,
    4: 0
}

y_true = df_eval["source_query_project"].map(label_map)
y_pred = df_eval["cross_val_category"].map(label_map)

# Remove any remaining NaNs just in case
mask = y_true.notna() & y_pred.notna()
y_true = y_true[mask].astype(int)
y_pred = y_pred[mask].astype(int)

# -----------------------------
# 6. Metrics
# -----------------------------
kappa = cohen_kappa_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
cm = confusion_matrix(y_true, y_pred)

# -----------------------------
# 7. Output results
# -----------------------------
print("\n=== Evaluation Results ===")
print(f"Cohen's Kappa: {kappa:.4f}")
print(f"Precision:     {precision:.4f}")
print(f"Recall:        {recall:.4f}")

print("\nConfusion Matrix:")
print(cm)

Rows used for evaluation: 1130

=== Evaluation Results ===
Cohen's Kappa: 0.8926
Precision:     0.9844
Recall:        0.9392

Confusion Matrix:
[[379  11]
 [ 45 695]]


In [5]:
from sklearn.metrics import cohen_kappa_score
import pandas as pd
df = pd.read_csv("./human_oracle_cross_validation.csv")

# ---- clean labels ----
y1 = df["human_agreement"].astype(str).str.strip()
y2 = df["agreement_flag"].astype(str).str.strip()

# ---- compute Cohen's Kappa ----
kappa = cohen_kappa_score(y1, y2)

print("Cohen's Kappa:", kappa)

Cohen's Kappa: 0.8162861384039019
